# Does a second model family help? Gemma vs Qwen LoRA

**This is an experiment, not a submission.** It answers one question: does a Gemma-family classifier add anything that a Qwen-family one does not?

**How it works**
- Each model trains on Open Pool passages only, so all 2,699 official rows stay unseen and act as the test set. That replaces five-fold training, which makes each run about five times cheaper.
- The Open Pool's flagged rows and any exact copies of official text are dropped. Its near-prompt flags are empty in the current version, so the notebook also measures topic overlap with the official prompts itself and reports it with the result.
- Blending any two models usually helps, so the fair control is **Qwen + Qwen with a different seed**. There are two seeds of each model, and the verdict compares cross-family pairs with same-family pairs of equal size.
- The final test also includes an n-gram model, because a gain only matters if it survives alongside the style models.

**How to run it**
1. Add the **Name That LLM** competition data and the [Name That LLM Open Pool](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool) dataset as inputs.
2. In the notebook settings choose **GPU T4 x2** and turn **Internet on** (Kaggle may ask you to verify a phone number).
3. Leave `SMOKE = True` and use *Run All*. This takes a few minutes and checks everything end to end.
4. Set `SMOKE = False`, then *Save Version > Save & Run All*. Expect about 5 hours of GPU quota; the notebook stops itself at 8.

**Why Gemma runs in fp32:** T4s have no bf16, and Gemma's activations overflow fp16 (tested on Gemma 3 270M: 11 of 18 layers failed). Each run also checks its precision on real passages before training, so a problem shows up in seconds.

**Reading the result:** the last cell prints a verdict. Qwen3-1.7B is deliberately larger than Gemma 3 1B, so if the blend still wins, that is strong evidence. If it does not, the smaller model may be part of the reason.


## 1. Settings

In [ ]:
# Start with SMOKE = True. Set it to False for the real run.
SMOKE = True

FAMILIES = {
    'qwen':  dict(repo='Qwen/Qwen3-1.7B-Base', dtype='fp16'),
    'gemma': dict(repo='unsloth/gemma-3-1b-pt', dtype='fp32'),   # same weights as google/gemma-3-1b-pt, no login. fp32 because Gemma overflows in fp16
}
SEEDS = [1, 2]
PER_CLASS = 40 if SMOKE else 2000     # training rows per model, 36,000 in total for the real run
BUDGET_MIN = 10 if SMOKE else 150     # training time limit for each run
TOTAL_HOURS = 1 if SMOKE else 8       # hard stop for the whole notebook
LIMIT_STEPS = 8 if SMOKE else 0
EVAL_ROWS = 540 if SMOKE else 0       # 0 means every official row


## 2. Check the environment and download the models

In [ ]:
import importlib.util, json, shutil, subprocess, sys, urllib.request
from pathlib import Path
if importlib.util.find_spec('peft') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft'], check=True)
import peft, torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def fetch(repo):
    """Download a model over plain HTTP and check each file's size, so the runs need no network."""
    target = Path('/tmp/models') / repo.replace('/', '--'); target.mkdir(parents=True, exist_ok=True)
    info = json.load(urllib.request.urlopen(f'https://huggingface.co/api/models/{repo}?blobs=true', timeout=60))
    for item in info['siblings']:
        name, size = item['rfilename'], item.get('size')
        if '/' in name or not name.endswith(('.json', '.safetensors', '.model', '.txt')): continue
        path = target / name
        if path.exists() and path.stat().st_size == size: continue
        with urllib.request.urlopen(f'https://huggingface.co/{repo}/resolve/main/{name}', timeout=120) as source, open(path, 'wb') as sink:
            shutil.copyfileobj(source, sink, 1 << 24)
        assert size is None or path.stat().st_size == size, f'{name} is incomplete'
    return str(target)

for family in FAMILIES.values():   # download now, so a network problem shows up in the first minutes
    try:
        family['path'] = fetch(family['repo'])
    except Exception as error:
        raise SystemExit(f"Could not download {family['repo']}. Is Internet switched on in the notebook settings? {error!r}")
    print('ready:', family['repo'])


## 3. Scripts
Each run happens in its own process, so one failure cannot take down the rest.

In [ ]:
%%writefile prepare.py
"""Build the training rows from the Open Pool and keep them apart from the official rows."""
import argparse, json, re, unicodedata, zlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True); ap.add_argument('--pool', required=True); ap.add_argument('--work', required=True)
ap.add_argument('--per-class', type=int, default=2000); ap.add_argument('--seed', type=int, default=0)
a = ap.parse_args()
work = Path(a.work); work.mkdir(parents=True, exist_ok=True)

official = pd.read_csv(Path(a.data) / 'train.csv'); official['text'] = official.text.fillna('').astype(str)
labels = sorted(official.model.unique()); assert len(labels) == 18, labels

FLAGS = ['flag_train_prompt_regen', 'flag_near_train_prompt', 'flag_gemini37_routing_risk', 'flag_kb_proxy']
pool = pd.read_parquet(a.pool, columns=['text', 'model', 'match_weight'] + FLAGS)
audit = {'pool_rows': int(len(pool))}
pool = pool[pool.model.isin(labels)].copy()
pool['text'] = pool.text.fillna('').astype(str).str.strip()   # line breaks stay: they are part of an author's style
keep = pool.text.str.len() > 0
for f in FLAGS:
    hit = pool[f].fillna(False).astype(bool); audit[f] = int(hit.sum()); keep &= ~hit
pool = pool[keep]

def norm(t): return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', t)).strip()
exact = pool.text.map(norm).isin(set(official.text.map(norm)))
audit['exact_official_matches_removed'] = int(exact.sum())
pool = pool[~exact].drop_duplicates(['text', 'model'])

rng = np.random.default_rng(a.seed); parts = []
for lab in labels:   # same number per model, favouring rows whose typography matches the official rows
    g = pool[pool.model == lab]; n = min(a.per_class, len(g))
    if n < 30: raise SystemExit(f'Only {n} usable pool rows for {lab}: check the Open Pool version')
    w = pd.to_numeric(g.match_weight, errors='coerce').fillna(0).clip(lower=0).to_numpy(float)
    keys = np.log(np.maximum(w, 1e-12)) + rng.gumbel(size=len(g))
    parts.append(g.iloc[np.argsort(-keys)[:n]][['text', 'model']])
train = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=a.seed).reset_index(drop=True)
train.to_parquet(work / 'train_pool.parquet', index=False)
audit['train_rows'] = int(len(train)); audit['rows_per_model'] = {k: int(v) for k, v in train.model.value_counts().items()}

def shingles(t, n=8):
    w = re.findall(r'\w+', t.lower()); return {zlib.crc32(' '.join(w[i:i + n]).encode()) for i in range(len(w) - n + 1)}
seen = set()
for t in train.text: seen |= shingles(t)
share = np.array([len(s & seen) / max(len(s), 1) for s in map(shingles, official.text)])
audit['official_rows_sharing_20pct_of_8grams'] = float((share > 0.2).mean())
audit['mean_8gram_overlap'] = float(share.mean())
# The near-prompt flags can be empty, so also measure directly whether the training rows cover the official prompts.
# 8-grams miss this: two answers to one prompt share a topic but few exact phrases.
vec = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
X = vec.fit_transform(pd.concat([official.text, train.text])); A, B = X[:len(official)], X[len(official):]
nearest = np.concatenate([(A[s:s + 256] @ B.T).max(axis=1).toarray().ravel() for s in range(0, A.shape[0], 256)])
sim = (A @ A.T).toarray(); np.fill_diagonal(sim, np.nan); same = official.prompt_id.to_numpy()[:, None] == official.prompt_id.to_numpy()[None, :]
same_prompt = np.nanmean(np.where(same, sim, np.nan), axis=1)   # how alike two answers to one prompt usually are
other_prompt = np.nanmax(np.where(same, np.nan, sim), axis=1)   # the closest answer to a different prompt
audit['topic_similarity_same_prompt_median'] = float(np.nanmedian(same_prompt))
audit['topic_similarity_nearest_other_prompt_median'] = float(np.nanmedian(other_prompt))
audit['topic_similarity_nearest_training_row_median'] = float(np.median(nearest))
audit['official_rows_on_a_prompt_the_training_rows_cover'] = float(np.nanmean(nearest > same_prompt))
(work / 'prepare.json').write_text(json.dumps(audit, indent=1))
print(json.dumps({k: v for k, v in audit.items() if k != 'rows_per_model'}, indent=1))
if audit['official_rows_sharing_20pct_of_8grams'] > 0.01:
    print('WARNING: over 1% of official rows closely overlap the training rows, so absolute scores may be inflated.')
if audit['official_rows_on_a_prompt_the_training_rows_cover'] > 0.10:
    print('WARNING: the training rows appear to cover many official prompts, so absolute scores may be inflated. The Qwen vs Gemma comparison is less affected.')


In [ ]:
%%writefile ngram.py
"""Style baseline: an n-gram SVC trained on the same pool rows, scored on the official rows."""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import log_softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True); ap.add_argument('--work', required=True); ap.add_argument('--eval-rows', type=int, default=0)
a = ap.parse_args()
work = Path(a.work); train = pd.read_parquet(work / 'train_pool.parquet')
official = pd.read_csv(Path(a.data) / 'train.csv'); official['text'] = official.text.fillna('').astype(str)
if a.eval_rows: official = official.head(a.eval_rows)
labels = sorted(pd.read_csv(Path(a.data) / 'train.csv').model.unique())
svc = Pipeline([('f', FeatureUnion([
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=300000, sublinear_tf=True, lowercase=False, dtype=np.float32)),
    ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=100000, sublinear_tf=True, lowercase=False, token_pattern=r'\w+|[^\w\s]', dtype=np.float32))])),
    ('clf', LinearSVC(C=0.5, class_weight='balanced', max_iter=5000))])
svc.fit(train.text, train.model)
d = svc.decision_function(official.text)[:, [list(svc.classes_).index(l) for l in labels]]
np.savez(work / 'ngram.npz', logp=log_softmax(d * 4, axis=1).astype(np.float32))
print('ngram done', d.shape, flush=True)


In [ ]:
%%writefile worker.py
"""Train one LoRA classifier on the pool rows, then score the official rows, which it never trains on."""
import argparse, importlib.util, json, math, time
from pathlib import Path

# Kaggle ships a torchao that peft refuses to load. Nothing here uses it, so hide it before peft looks.
_find_spec = importlib.util.find_spec
importlib.util.find_spec = lambda name, *args, **kw: None if name.split('.')[0] == 'torchao' else _find_spec(name, *args, **kw)

import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model

ap = argparse.ArgumentParser()
ap.add_argument('--name', required=True); ap.add_argument('--repo', required=True); ap.add_argument('--dtype', choices=['fp16', 'fp32'], required=True)
ap.add_argument('--seed', type=int, required=True); ap.add_argument('--data', required=True); ap.add_argument('--work', required=True)
ap.add_argument('--budget-min', type=float, default=150); ap.add_argument('--deadline', type=float, default=0)
ap.add_argument('--bs', type=int, default=8); ap.add_argument('--accum', type=int, default=4); ap.add_argument('--max-len', type=int, default=384)
ap.add_argument('--lr', type=float, default=1e-4); ap.add_argument('--rank', type=int, default=16)
ap.add_argument('--limit-steps', type=int, default=0); ap.add_argument('--eval-rows', type=int, default=0)
a = ap.parse_args()

T0 = time.time()
def log(*x): print(f'[{(time.time() - T0) / 60:7.1f}m {a.name}]', *x, flush=True)
out = Path(a.work) / 'runs' / a.name; out.mkdir(parents=True, exist_ok=True)
status = dict(name=a.name, repo=a.repo, dtype=a.dtype, seed=a.seed, status='running')
def save_status(**kw):
    status.update(kw); (out / 'run.json').write_text(json.dumps(status, indent=1))
save_status()

torch.manual_seed(a.seed); rng = np.random.default_rng(a.seed)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
amp = a.dtype == 'fp16' and dev == 'cuda'
train = pd.read_parquet(Path(a.work) / 'train_pool.parquet')
official = pd.read_csv(Path(a.data) / 'train.csv'); official['text'] = official.text.fillna('').astype(str)
labels = sorted(official.model.unique()); to_id = {l: i for i, l in enumerate(labels)}
if a.eval_rows: official = official.head(a.eval_rows)
y = train.model.map(to_id).to_numpy()

tok = AutoTokenizer.from_pretrained(a.repo)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'right'
enc = tok(train.text.tolist(), truncation=True, max_length=a.max_len)['input_ids']
enc_off = tok(official.text.tolist(), truncation=True, max_length=a.max_len)['input_ids']
lens = np.array([len(x) for x in enc]); log('rows', len(enc), 'mean tokens', round(float(lens.mean())))

base = AutoModel.from_pretrained(a.repo, dtype=torch.float16 if a.dtype == 'fp16' else torch.float32, attn_implementation='sdpa').to(dev)
n_layers = base.config.num_hidden_layers; hidden = base.config.hidden_size
picks = sorted({n_layers // 2, 3 * n_layers // 4, n_layers})
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model = get_peft_model(base, LoraConfig(r=a.rank, lora_alpha=2 * a.rank, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))

class Head(nn.Module):
    def __init__(self):
        super().__init__(); self.norms = nn.ModuleList(nn.LayerNorm(hidden) for _ in picks); self.out = nn.Linear(hidden * len(picks), 18)
    def features(self, pooled): return torch.cat([n(p) for n, p in zip(self.norms, pooled)], -1)
    def forward(self, pooled): return self.out(self.features(pooled))
head = Head().to(dev)

def batch(rows, ids):
    return tok.pad({'input_ids': [ids[int(i)] for i in rows]}, return_tensors='pt').to(dev)
def pool_layers(b):   # mean over real tokens, skipping position 0, which most decoders use as an attention sink
    with torch.autocast(dev, dtype=torch.float16, enabled=amp):
        hs = model(input_ids=b['input_ids'], attention_mask=b['attention_mask'], output_hidden_states=True, use_cache=False).hidden_states
    m = b['attention_mask'].clone().float(); m[:, 0] = 0; m = m.unsqueeze(-1); den = m.sum(1).clamp(min=1)
    return [(hs[l].float() * m).sum(1) / den for l in picks]

model.eval()
with torch.inference_mode():   # fail in seconds, not hours, if this precision overflows on real passages
    probe = pool_layers(batch(np.argsort(-lens)[:16], enc))
if not all(torch.isfinite(p).all() for p in probe):
    save_status(status='failed', reason=f'non-finite activations in {a.dtype}'); raise SystemExit(f'{a.name}: activations overflow in {a.dtype}')
model.train(); head.train()

lora = [p for p in model.parameters() if p.requires_grad]; log('trainable', sum(p.numel() for p in lora), 'adapter dtype', lora[0].dtype, 'layers', picks)
opt = torch.optim.AdamW([{'params': lora, 'lr': a.lr, 'weight_decay': 0.0}, {'params': head.parameters(), 'lr': a.lr * 10, 'weight_decay': 0.01}])
peak = [g['lr'] for g in opt.param_groups]
scaler = torch.amp.GradScaler('cuda', enabled=amp)

order = rng.permutation(len(enc)); micro = []
for s in range(0, len(order), a.bs * 64):   # group similar lengths so little compute goes on padding
    chunk = order[s:s + a.bs * 64]; chunk = chunk[np.argsort(lens[chunk], kind='stable')]
    micro += [chunk[i:i + a.bs] for i in range(0, len(chunk), a.bs)]
micro = [micro[i] for i in rng.permutation(len(micro))]
total = math.ceil(len(micro) / a.accum)
if a.limit_steps: total = min(total, a.limit_steps)
budget = a.budget_min * 60
if a.deadline: budget = min(budget, a.deadline - time.time() - 300)
if budget <= 0: save_status(status='failed', reason='no time left'); raise SystemExit(f'{a.name}: no time left')

def set_lr(step):
    warm = max(1, int(0.03 * total))
    mult = (step + 1) / warm if step < warm else 0.1 + 0.45 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    for g, p in zip(opt.param_groups, peak): g['lr'] = p * mult

step = seen = bad = 0; t_train = time.time(); t_measure = None; recent = []; loss_sum = 0.0
for i, rows in enumerate(micro):
    if step >= total: break
    b = batch(rows, enc); target = torch.as_tensor(y[rows], device=dev)
    loss = F.cross_entropy(head(pool_layers(b)), target)
    if not torch.isfinite(loss):
        bad += 1
        if bad >= 10: save_status(status='failed', reason='loss went non-finite'); raise SystemExit(f'{a.name}: loss went non-finite')
        continue
    scaler.scale(loss / a.accum).backward(); seen += len(rows); loss_sum += loss.item()
    if (i + 1) % a.accum: continue
    scaler.unscale_(opt); nn.utils.clip_grad_norm_(lora + list(head.parameters()), 1.0)
    set_lr(step); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
    recent.append(loss_sum / a.accum); loss_sum = 0.0
    if step == 3: t_measure = time.time()
    if step == 23 and t_measure and not a.limit_steps:   # fit the schedule to the time budget, so the learning rate still decays
        per = (time.time() - t_measure) / 20; fit = step + int((budget - (time.time() - t_train)) / per)
        if fit < total: log(f'budget allows {fit} of {total} steps'); total = fit
        log(f'{per:.2f}s per step; about {total * per / 60:.0f} minutes to train')
    if step % 100 == 0: log(f'step {step}/{total} rows {seen} loss {np.mean(recent[-100:]):.3f}'); save_status(step=step, rows_seen=seen)
train_min = (time.time() - t_train) / 60

model.eval(); head.eval(); N = len(enc_off); logp = np.zeros((N, 18), np.float32); feats = None
with torch.inference_mode():
    idx = np.argsort([len(x) for x in enc_off], kind='stable')
    for s in range(0, N, 16):
        rows = idx[s:s + 16]; f = head.features(pool_layers(batch(rows, enc_off)))
        logp[rows] = torch.log_softmax(head.out(f), -1).cpu().numpy()
        if feats is None: feats = np.zeros((N, f.shape[1]), np.float16)
        feats[rows] = f.clamp(-60000, 60000).cpu().numpy()
assert np.isfinite(logp).all(), 'non-finite predictions'
np.savez(out / 'pred.npz', logp=logp, feats=feats)
model.save_pretrained(out / 'adapter'); torch.save(head.state_dict(), out / 'head.pt')
true = official.model.map(to_id).to_numpy(); pred = logp.argmax(1)
score = float(np.mean([(pred[true == k] == k).mean() for k in range(18) if (true == k).any()]))
peak_gb = round(torch.cuda.max_memory_allocated() / 2**30, 2) if dev == 'cuda' else None
save_status(status='done', step=step, planned_steps=total, rows_seen=seen, train_minutes=round(train_min, 1), final_loss=round(float(np.mean(recent[-50:])), 4), official_score=round(score, 4), peak_gb=peak_gb)
log('done', json.dumps(status))


In [ ]:
%%writefile report.py
"""Compare the runs on the official rows. The test is cross-family pairs against same-family pairs of equal size."""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True); ap.add_argument('--work', required=True)
a = ap.parse_args()
work = Path(a.work)
off = pd.read_csv(Path(a.data) / 'train.csv'); labels = sorted(off.model.unique()); to_id = {l: i for i, l in enumerate(labels)}
runs = {}; info = {}
for d in sorted((work / 'runs').glob('*')):
    meta = json.loads((d / 'run.json').read_text()) if (d / 'run.json').exists() else {'status': 'missing'}
    info[d.name] = meta
    if meta.get('status') == 'done' and (d / 'pred.npz').exists(): runs[d.name] = np.load(d / 'pred.npz')['logp'].astype(np.float64)
ng = np.load(work / 'ngram.npz')['logp'].astype(np.float64) if (work / 'ngram.npz').exists() else None
if not runs: raise SystemExit('No finished runs: read the worker logs above.')
n = next(iter(runs.values())).shape[0]; off = off.head(n)
y = off.model.map(to_id).to_numpy(); g = off.prompt_id.to_numpy(); prompts = np.unique(g); where = {p: np.where(g == p)[0] for p in prompts}

def score(p, rows=slice(None)):
    pred, t = p.argmax(1)[rows], y[rows]
    return float(np.mean([(pred[t == k] == k).mean() for k in range(18) if (t == k).any()]))
blend = lambda *m: sum(m) / len(m)
rng = np.random.default_rng(0); samples = [np.concatenate([where[p] for p in rng.choice(prompts, len(prompts))]) for _ in range(2000)]
def compare(tests, control):
    d = np.array([np.mean([score(t, s) for t in tests]) - score(control, s) for s in samples])
    return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

lines = ['## Single models', '', '| Run | Score | Rows seen | Minutes | Final loss |', '|---|---|---|---|---|']
for k, m in info.items():
    if k in runs: lines.append(f"| {k} | {score(runs[k]):.4f} | {m.get('rows_seen')} | {m.get('train_minutes')} | {m.get('final_loss')} |")
    else: lines.append(f"| {k} | FAILED: {m.get('reason', m.get('status'))} | | | |")
if ng is not None: lines.append(f'| n-gram SVC | {score(ng):.4f} | | | |')
seen = {m.get('rows_seen') for k, m in info.items() if k in runs}
if len(seen) > 1: lines += ['', f'**Note:** runs saw different numbers of rows ({sorted(seen)}), so single-model scores are not like for like.']

q, m = sorted(k for k in runs if k.startswith('qwen')), sorted(k for k in runs if k.startswith('gemma'))
verdict = 'Not enough finished runs for the comparison: it needs two seeds of each model.'
out = {'single': {k: score(v) for k, v in runs.items()}, 'ngram': None if ng is None else score(ng)}
if len(q) >= 2 and len(m) >= 2:
    Q1, Q2, G1, G2 = runs[q[0]], runs[q[1]], runs[m[0]], runs[m[1]]
    lines += ['', '## Pairs', '', '| Pair | Score |', '|---|---|', f'| Qwen + Qwen | {score(blend(Q1, Q2)):.4f} |', f'| Gemma + Gemma | {score(blend(G1, G2)):.4f} |',
              f'| Qwen + Gemma (first seeds) | {score(blend(Q1, G1)):.4f} |', f'| Qwen + Gemma (second seeds) | {score(blend(Q2, G2)):.4f} |']
    tests = [('Cross-family pair vs Qwen + Qwen', [blend(Q1, G1), blend(Q2, G2)], blend(Q1, Q2))]
    if ng is not None:
        tests.append(('With n-gram: cross-family vs Qwen + Qwen', [blend(ng, Q1, G1), blend(ng, Q2, G2)], blend(ng, Q1, Q2)))
    lines += ['', '## The test: cross-family minus same-family (95% interval over prompts)', '', '| Comparison | Difference | Interval |', '|---|---|---|']
    res = {}
    for name, t, c in tests:
        res[name] = compare(t, c); lines.append(f'| {name} | {res[name][0]:+.4f} | {res[name][1]:+.4f} to {res[name][2]:+.4f} |')
    key = res[tests[-1][0]]; out['tests'] = res
    if key[1] > 0: verdict = 'Gemma adds something Qwen does not: the interval is above zero. Worth scaling up to a larger Gemma.'
    elif key[2] < 0: verdict = 'A second family made things worse here: the interval is below zero. Do not scale this up yet.'
    else: verdict = 'Inconclusive: the interval includes zero, so this run cannot separate a real gain from noise.'
prep = json.loads((work / 'prepare.json').read_text())
cover = prep.get('official_rows_on_a_prompt_the_training_rows_cover')
lines += ['', f'**Verdict:** {verdict}', '', 'Scores are on all official rows, which no model trained on. Blends use equal weights.',
          f"Overlap check: {cover:.1%} of official rows look like they sit on a prompt the training rows cover; {prep['official_rows_sharing_20pct_of_8grams']:.1%} share over 20% of their 8-grams."]
(work / 'report.md').write_text('\n'.join(lines) + '\n'); out['verdict'] = verdict; (work / 'report.json').write_text(json.dumps(out, indent=1))
print('\n'.join(lines))


## 4. Run
Qwen trains on one GPU and Gemma on the other, at the same time. The n-gram model runs on the CPU alongside them.

In [ ]:
import glob, os, threading, time
from pathlib import Path

DATA = next((p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv')), None)
assert DATA, 'Add the Name That LLM competition data as an input.'
pools = sorted(glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True))
assert pools, 'Add the Name That LLM Open Pool dataset as an input.'
WORK = '/kaggle/working/run'
LOGS = Path(WORK) / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)
GPUS = list(range(torch.cuda.device_count()))
assert GPUS, 'No GPU found: choose GPU T4 x2 in the notebook settings.'
if len(GPUS) < 2: print('WARNING: one GPU found, so runs go one at a time and the later ones may hit the time limit.')
deadline = time.time() + TOTAL_HOURS * 3600

subprocess.run([sys.executable, 'prepare.py', '--data', DATA, '--pool', pools[0], '--work', WORK, '--per-class', str(PER_CLASS)], check=True)

failed = []
queues = {i: [] for i in range(len(GPUS))}
for n, (family, cfg) in enumerate(FAMILIES.items()):
    queues[n % len(GPUS)] += [(f'{family}_s{seed}', cfg, seed) for seed in SEEDS]

def run_queue(slot):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPUS[slot]), TOKENIZERS_PARALLELISM='false')
    for name, cfg, seed in queues[slot]:
        command = [sys.executable, 'worker.py', '--name', name, '--repo', cfg['path'], '--dtype', cfg['dtype'], '--seed', str(seed),
                   '--data', DATA, '--work', WORK, '--budget-min', str(BUDGET_MIN), '--deadline', str(deadline),
                   '--limit-steps', str(LIMIT_STEPS), '--eval-rows', str(EVAL_ROWS)]
        with open(LOGS / f'{name}.log', 'w') as handle:
            code = subprocess.run(command, env=env, stdout=handle, stderr=subprocess.STDOUT).returncode
        print(f'{name} finished, exit code {code}', flush=True)
        if code: failed.append(name)

def run_ngram():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='')
    with open(LOGS / 'ngram.log', 'w') as handle:
        code = subprocess.run([sys.executable, 'ngram.py', '--data', DATA, '--work', WORK, '--eval-rows', str(EVAL_ROWS)], env=env, stdout=handle, stderr=subprocess.STDOUT).returncode
    print(f'n-gram finished, exit code {code}', flush=True)
    if code: failed.append('ngram')

threads = [threading.Thread(target=run_queue, args=(slot,)) for slot in queues] + [threading.Thread(target=run_ngram)]
for t in threads: t.start()
while any(t.is_alive() for t in threads):
    time.sleep(30 if SMOKE else 600)
    for log in sorted(LOGS.glob('*.log')):
        lines = log.read_text(errors='replace').strip().splitlines()
        if lines: print(log.stem, '|', lines[-1][:120], flush=True)
for t in threads: t.join()

for name in failed:   # show why anything failed
    print(f'--- {name} failed ---', *(LOGS / f'{name}.log').read_text(errors='replace').strip().splitlines()[-15:], sep='\n')


## 5. Result

In [ ]:
subprocess.run([sys.executable, 'report.py', '--data', DATA, '--work', WORK], check=True, stdout=subprocess.DEVNULL)
print('SMOKE TEST ONLY: these numbers mean nothing. Set SMOKE = False for the real run.\n' if SMOKE else '')
print(open(f'{WORK}/report.md').read())
